In [19]:
import json
from typing import Annotated, Any, Dict, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from google.colab import userdata
from uuid import uuid4

In [20]:
@tool
def get_server_health(server_id: str) -> str:
    """Checks CPU, memory, and status for a given server."""
    print(f"-> TOOL: Checking health for {server_id}...")

    metrics = {
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},
        "db-node-02": {"cpu": "12%", "memory": "60%", "status": "Healthy"},
        "auth-service-03": {"cpu": "45%", "memory": "95%", "status": "Critical"},
        "search-index-09": {"cpu": "10%", "memory": "15%", "status": "Error"},
        "frontend-node-04": {"cpu": "25%", "memory": "30%", "status": "Healthy"},
    }

    return json.dumps(metrics.get(server_id, {"error": "Server not found. Check the ID."}))


@tool
def fetch_recent_logs(server_id: str, lines: int = 5) -> str:
    """Fetches recent log lines from a given server."""
    print(f"-> TOOL: Fetching last {lines} log lines for {server_id}...")

    log_database = {
        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread",
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active",
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context...",
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s...",
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed",
        ],
    }

    logs = log_database.get(server_id, ["[INFO] System stable", "[INFO] Heartbeat signal received"])
    return json.dumps({"logs": logs[:lines]})


@tool
def restart_service(server_id: str) -> str:
    """Restarts a service for a known server."""
    print(f"-> TOOL: Restarting service for {server_id}...")

    server_list = [
        "payment-server-01",
        "db-node-02",
        "auth-service-03",
        "frontend-node-04",
    ]

    if server_id not in server_list:
        return json.dumps({"status": "skipped", "message": "Server is not restartable or not found."})

    return json.dumps({"status": "success", "message": "Server restarted successfully."})


@tool
def escalate_to_engineer(summary: str) -> str:
    """Escalates the incident to a human engineer using a JSON summary."""
    print("-> TOOL: Escalating to human...")
    return json.dumps({
        "status": "success",
        "message": "Ticket created successfully.",
        "ticket_id": str(uuid4()),
        "escalation_type": "human",
        "summary": summary,
    })


TOOLS = [get_server_health, fetch_recent_logs, restart_service, escalate_to_engineer]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

In [21]:
class ITAgentState(TypedDict):
    # add_messages appends new messages instead of overwriting the list.
    messages: Annotated[list, add_messages]
    collected_data: Dict[str, Any]


SYSTEM_PROMPT = """
You are a Level 1 IT Responder. Investigate server issues.

Rules:
1. Always inspect server health and recent logs before deciding.
2. If CPU or Memory is greater than 90%, restart the service.
3. If logs show dependency/network failures such as connection refused, dependency unreachable, or external service down, escalate to an engineer.
4. If the server is healthy and logs are normal, do not restart or escalate. Summarize findings.
5. When escalating, pass a JSON summary containing health, logs, restart result if any, and the reason for escalation.
"""

In [22]:
llm = ChatOpenAI(model="gpt-4o", temperature=0, api_key=userdata.get('openai_IK')).bind_tools(TOOLS)


def call_model(state: ITAgentState) -> Dict[str, Any]:
    """LLM reasoning node. It may respond directly or request tool calls."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


def execute_tools(state: ITAgentState) -> Dict[str, Any]:
    """Executes tool calls requested by the model and updates collected_data."""
    last_message: AIMessage = state["messages"][-1]
    collected_data = dict(state.get("collected_data", {}))
    tool_messages = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = dict(tool_call.get("args", {}))
        tool_id = tool_call["id"]

        # Match the original notebook behavior: inject collected_data into escalation.
        if tool_name == "escalate_to_engineer":
            tool_args = {"summary": json.dumps(collected_data)}

        selected_tool = TOOLS_BY_NAME[tool_name]
        tool_output = selected_tool.invoke(tool_args)

        try:
            parsed_output = json.loads(tool_output)
        except Exception:
            parsed_output = tool_output

        if tool_name == "get_server_health":
            collected_data["server_health"] = parsed_output
        elif tool_name == "fetch_recent_logs":
            collected_data["server_logs"] = parsed_output
        elif tool_name == "restart_service":
            collected_data["restart_logs"] = parsed_output
        elif tool_name == "escalate_to_engineer":
            collected_data["escalation"] = parsed_output

        tool_messages.append(
            ToolMessage(
                content=tool_output,
                name=tool_name,
                tool_call_id=tool_id,
            )
        )

    return {
        "messages": tool_messages,
        "collected_data": collected_data,
    }


def should_continue(state: ITAgentState) -> str:
    """Routes to tool execution if the LLM requested tools, otherwise ends."""
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tools"
    return END

In [23]:
graph_builder = StateGraph(ITAgentState)
graph_builder.add_node("agent", call_model)
graph_builder.add_node("tools", execute_tools)

graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph_builder.add_edge("tools", "agent")

it_support_graph = graph_builder.compile()


In [32]:
def run_it_agent(user_issue: str) -> str:
    print(f"\n--- New Incident: {user_issue} ---")

    initial_state = {
        "messages": [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=user_issue),
        ],
        "collected_data": {},
    }

    result = it_support_graph.invoke(initial_state)
    final_message = result["messages"][-1].content

    print(f"\n[FINAL RESPONSE]: {final_message}")
    return final_message


In [33]:
while True:
    user_issue = input("Enter your issue: ")

    if user_issue.lower() == "exit" or user_issue.lower() == 'q':
      print("AI shutdown !!")
      break

    run_it_agent(user_issue)
    print("\n" + "=" * 50)

Enter your issue: Search isn't working. Can you check search-index-09?

--- New Incident: Search isn't working. Can you check search-index-09? ---
-> TOOL: Checking health for search-index-09...
-> TOOL: Fetching last 5 log lines for search-index-09...
-> TOOL: Escalating to human...

[FINAL RESPONSE]: The search-index-09 server is experiencing issues related to dependency and network failures. The logs indicate connection refused errors and a critical dependency being unreachable. I've escalated this issue to an engineer for further investigation. A ticket has been created with ID: f7f29ac3-5599-4177-9ae8-eb6f4a82da58.

Enter your issue: q
AI shutdown !!
